# Setup

## Install dependencies

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !uv pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !uv pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !uv pip install --no-deps unsloth
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2
!uv pip install -q wandb evaluate sacrebleu rouge_score bert_score nltk

## Global variable configuration

In [ ]:
# Model settings
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1500
DTYPE = None  
LOAD_IN_4BIT = True 

# LoRA settings
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

BATCH_SIZE = 12
GRADIENT_ACCUMULATION_STEPS = 4
NUM_EPOCHS = 3
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 5

# Paths & Hub
DATASET_PATH = "quannguyen204/medical_sft_crawl_vi_10k_v1"
OUTPUT_DIR = "./outputs/sft"
HUB_MODEL_ID = "quannguyen204/vimed-llama3.2-3b-sft-v1"

# W&B
WANDB_PROJECT = "ViMed-Assistant-DPO"
WANDB_RUN_NAME = "sft-llama3.2-3b-med10k-v1"

## Authentication

In [ ]:
import wandb
from huggingface_hub import login

# HuggingFace login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# W&B login
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: minhquana (minhquana-university-of-transportation-and-communication) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Model loaded: {BASE_MODEL}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Max seq length: {MAX_SEQ_LENGTH}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-03 16:56:37.129515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764780997.495285      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764780997.611941      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find Trainer class in trl.trainer.bco_trainer. Found: ['BCOTrainer', '_BCOTrainer']
==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Model loaded: unsloth/Llama-3.2-3B-Instruct-bnb-4bit
Vocab size: 128256
Max seq length: 1500


## Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Reduce VRAM 30%
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

# Print trainable parameters
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.11.6 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 48,627,712 || all params: 3,261,377,536 || trainable%: 1.4910


Apply suitable chat template for Llama3.2

In [8]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.2",
)

## Data configuration

### Load dataset from Huggingface

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_PATH)

print(f"Dataset splits: {list(dataset.keys())}")
print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples: {len(dataset['test'])}")
print(f"Columns: {dataset['train'].column_names}")

# Preview sample
print("\n--- Sample ---")
print(dataset['train'][0])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.72M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/634k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9076 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1009 [00:00<?, ? examples/s]

Dataset size: 9076
Columns: ['messages']

--- Sample ---
{'messages': [{'content': 'Bạn là một trợ lý y tế ảo thông minh, với vai trò là một bác sĩ tư vấn trực tuyến chuyên nghiệp và tận tâm. Nhiệm vụ của bạn là giải đáp thắc mắc, câu hỏi về chủ đề y tế. Câu trả lời cần mang tính định hướng, giải thích nguyên nhân có thể, không được thay thế chẩn đoán của bệnh viện và phải luôn khuyên người dùng đến cơ sở y tế để có chẩn đoán chính xác.', 'role': 'system'}, {'content': 'Liệt kê các phương pháp điều trị viêm tụy tự miễn?', 'role': 'user'}, {'content': 'Viêm tụy tự miễn được điều trị bằng corticosteroid, thường bắt đầu với 40 mg/ngày prednisone mỗi ngày trong bốn tuần. Tình trạng lâm sàng của bệnh nhân sau đó được đánh giá lại và các nghiên cứu huyết thanh và X-quang được lặp lại. Nếu đáp ứng phù hợp, liều lượng được giảm dần 5mg/tuần cho đến khi hoàn thành. Azathioprine hoặc rituximab được sử dụng cho những bệnh nhân có chống chỉ định dùng steroid hoặc để điều trị các đợt tái phát.', 'r

## Prepare data

In [ ]:
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]  
test_dataset = dataset["test"]  

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples (for training): {len(eval_dataset)}")
print(f"Test samples (for final evaluation): {len(test_dataset)}")

Train samples: 8168
Eval samples: 908


## Trainer Configuration

### Setup SFTTrainer with train_on_responses_only

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import train_on_responses_only
from transformers import EarlyStoppingCallback, DataCollatorForSeq2Seq

# SFT Config
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Training hyperparams
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=1.0,

    # Optimizer
    optim="adamw_8bit",
    lr_scheduler_type="cosine",

    # Precision
    fp16=True,
    bf16=False,

    # Evaluation & Saving
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=1,
    report_to="wandb",
    run_name=WANDB_RUN_NAME,

    # Dataset
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field=None, 
    packing=False,  

    # Misc
    seed=42,
    dataloader_num_workers=4,
)

### Formatting Function for Conversational Data

In [ ]:
def formatting_prompts_func(examples):
    """
    Apply chat template to conversations.
    Data format: {"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}
    """
    convos = examples["messages"]

    if isinstance(convos[0], dict):
        text = tokenizer.apply_chat_template(
            convos,
            tokenize=False,
            add_generation_prompt=False
        )
        return [text]

    texts = []
    for messages in convos:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return texts

### Initialize Trainer

In [ ]:
# Initialize W&B
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "model": BASE_MODEL,
        "lora_r": LORA_R,
        "learning_rate": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "max_seq_length": MAX_SEQ_LENGTH,
    }
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    formatting_func=formatting_prompts_func,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)


print("Trainer initialized with train_on_responses_only")

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["None"] (num_proc=8):   0%|          | 0/8168 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["None"] (num_proc=8):   0%|          | 0/908 [00:00<?, ? examples/s]

Trainer initialized with train_on_responses_only


In [ ]:
# Verify Masking (Optional Debug)
sample = train_dataset[0]
formatted_output = formatting_prompts_func({"messages": [sample["messages"]]})

raw_text = formatted_output[0]

print("Sample text preview:")
print(raw_text)

tokenized = tokenizer(
    raw_text,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_SEQ_LENGTH
)

print(f"\nTokenized shape: {tokenized.input_ids.shape}")

Sample text preview:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Bạn là một trợ lý y tế ảo thông minh, với vai trò là một bác sĩ tư vấn trực tuyến chuyên nghiệp và tận tâm. Nhiệm vụ của bạn là giải đáp thắc mắc, câu hỏi về chủ đề y tế. Câu trả lời cần mang tính định hướng, giải thích nguyên nhân có thể, không được thay thế chẩn đoán của bệnh viện và phải luôn khuyên người dùng đến cơ sở y tế để có chẩn đoán chính xác.<|eot_id|><|start_header_id|>user<|end_header_id|>

Những dấu hiệu nào cho thấy một người đang bị hạ thân nhiệt?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Khi nhiệt độ cơ thể giảm xuống, tim, hệ thần kinh và các cơ quan khác không thể hoạt động bình thường. Nếu không được điều trị, hạ thân nhiệt có thể dẫn đến suy tim và hệ hô hấp hoàn toàn, cuối cùng dẫn đến tử vong.<|eot_id|>

Tokenized shape: torch.Size([1, 205])


In [ ]:
# GPU Memory Check Before Training
import torch

def print_gpu_memory():
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        reserved = torch.cuda.memory_reserved(i) / 1024**3
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"GPU {i}: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved, {total:.2f}GB total")

print_gpu_memory()

GPU 0: 2.29GB allocated, 2.29GB reserved, 14.74GB total
GPU 1: 0.02GB allocated, 0.02GB reserved, 14.74GB total


# Training

In [ ]:
print("Starting SFT training...")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Total training samples: {len(train_dataset)}")
print(f"Epochs: {NUM_EPOCHS}")

# Train
trainer_stats = trainer.train()

print("\n--- Training Complete ---")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Training runtime: {trainer_stats.metrics['train_runtime']:.2f}s")

Starting SFT training...
Effective batch size: 16
Total training samples: 8168
Epochs: 3


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,168 | Num Epochs = 3 | Total steps = 1,533
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,0.596500,1.258862
100,0.582300,1.199259
150,0.462800,1.154585
200,0.449700,1.134097
250,0.480400,1.122436
300,0.441500,1.104226
350,0.408700,1.094345
400,0.450800,1.076304
450,0.446200,1.062586
500,0.458100,1.048129


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


eval/loss,█▇▅▅▅▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/runtime,▄▄▁▄▁▇▅▆▇▆▁▃▅▅█▆▆▅▃█▅▃▆▆▃▂▄▁▅▃
eval/samples_per_second,▅▅█▅█▂▄▃▂▃█▆▄▄▁▃▃▄▆▁▄▆▃▃▆▇▅█▅▆
eval/steps_per_second,▅▅█▅█▂▄▄▂▃█▆▄▄▁▃▃▄▆▁▄▅▃▃▆▇▅█▄▆
train/epoch,▁▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train/grad_norm,█▂▁▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▃▃▂▃▃▃▃
train/learning_rate,▃▄▇▇▇████████▇▇▇▆▆▆▆▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁▁
train/loss,▅▄▄▅█▅▄█▄▂▅▇▄▄█▆▃▁▃▄▄▇▆▅▄▃▃▃▂▁▆▂▆▁▄▂▅▄▃▂
eval/loss,0.96895
eval/runtime,234.5819



--- Training Complete ---
Training loss: 0.9759
Training runtime: 21232.45s


# Evaluation on Test Set

This section evaluates model generalization using metrics:
- BLEU: N-gram overlap (precision-based)
- ROUGE-L: Longest common subsequence (recall-based)
- BERTScore: Semantic similarity using BERT embeddings

In [ ]:
import evaluate
import numpy as np
from tqdm import tqdm

# Load evaluation metrics
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("Evaluation metrics loaded successfully")

In [ ]:
NUM_EVAL_SAMPLES = min(300, len(test_dataset))  
eval_indices = np.random.choice(len(test_dataset), NUM_EVAL_SAMPLES, replace=False)
eval_subset = test_dataset.select(eval_indices)

print(f"Evaluating on {NUM_EVAL_SAMPLES} samples from test set...")

## Run Inference and Collect Predictions

In [ ]:
predictions = []
references = []

for sample in tqdm(eval_subset, desc="Generating predictions"):
    messages = sample["messages"]
    
    # Extract ground truth (last assistant message)
    ground_truth = None
    prompt_messages = []
    
    for msg in messages:
        if msg["role"] == "assistant":
            ground_truth = msg["content"]
        else:
            prompt_messages.append(msg)
    
    if ground_truth is None:
        continue
    
    try:
        pred = generate_response(prompt_messages)
        predictions.append(pred)
        references.append(ground_truth)
    except Exception as e:
        print(f"Error generating response: {e}")
        continue

print(f"\nGenerated {len(predictions)} predictions")

## Compute Evaluation Metrics

In [ ]:
print("Computing evaluation metrics...")

# BLEU Score - Measures n-gram precision
# Higher is better, range [0, 100]
bleu_results = bleu_metric.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

# ROUGE Scores - Measures recall of n-grams and longest common subsequence
# Higher is better, range [0, 1]
rouge_results = rouge_metric.compute(
    predictions=predictions,
    references=references
)

# BERTScore - Measures semantic similarity using BERT embeddings
# Higher is better, range [0, 1]
# Using multilingual model for Vietnamese
bertscore_results = bertscore_metric.compute(
    predictions=predictions,
    references=references,
    lang="vi",  
    model_type="bert-base-multilingual-cased"
)

# Aggregate BERTScore (mean over all samples)
bertscore_f1 = np.mean(bertscore_results["f1"])
bertscore_precision = np.mean(bertscore_results["precision"])
bertscore_recall = np.mean(bertscore_results["recall"])

## Log evaluation results

In [ ]:
print("\n" + "=" * 60)
print("SFT MODEL EVALUATION RESULTS (Test Set)")
print("=" * 60)

print(f"\nBLEU Score: {bleu_results['score']:.2f}")
print("   (Measures n-gram precision, higher is better)")

print(f"\nROUGE Scores:")
print(f"   ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"   ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"   ROUGE-L: {rouge_results['rougeL']:.4f}")
print("   (Measures recall of n-grams, higher is better)")

print(f"\nBERTScore (Semantic Similarity):")
print(f"   Precision: {bertscore_precision:.4f}")
print(f"   Recall:    {bertscore_recall:.4f}")
print(f"   F1:        {bertscore_f1:.4f}")
print("   (Measures semantic similarity, higher is better)")

print("\n" + "=" * 60)

# Log to W&B
eval_metrics = {
    "test/bleu": bleu_results['score'],
    "test/rouge1": rouge_results['rouge1'],
    "test/rouge2": rouge_results['rouge2'],
    "test/rougeL": rouge_results['rougeL'],
    "test/bertscore_f1": bertscore_f1,
    "test/bertscore_precision": bertscore_precision,
    "test/bertscore_recall": bertscore_recall,
    "test/num_samples": len(predictions),
}
wandb.log(eval_metrics)

print("\nMetrics logged to W&B")

In [ ]:
# Sample Predictions vs Ground Truth
print("\n" + "=" * 60)
print("SAMPLE PREDICTIONS VS GROUND TRUTH")
print("=" * 60)

# Show 3 random examples
sample_indices = np.random.choice(len(predictions), min(3, len(predictions)), replace=False)

for i, idx in enumerate(sample_indices):
    print(f"\n{'─' * 60}")
    print(f"Example {i+1}:")
    print(f"{'─' * 60}")
    print(f"Ground Truth:\n{references[idx][:500]}...")
    print(f"\nPrediction:\n{predictions[idx][:500]}...")
    print()

## Save model

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")

print(f"LoRA adapter saved to {OUTPUT_DIR}/lora_adapter")

LoRA adapter saved to ./outputs/sft/lora_adapter


## Push to Huggingface Hub

In [ ]:
model.push_to_hub(
    HUB_MODEL_ID,
    token=hf_token,
    private=False,
)
tokenizer.push_to_hub(
    HUB_MODEL_ID,
    token=hf_token,
)

print(f"Model pushed to https://huggingface.co/{HUB_MODEL_ID}")

README.md:   0%|          | 0.00/602 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/quannguyen204/vimed-llama3.2-3b-sft-v1


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [ ]:
# Save Merged 16-bit Model for DPO
model.save_pretrained_merged(
    f"{OUTPUT_DIR}/merged_16bit",
    tokenizer,
    save_method="merged_16bit",
)
print(f"Merged 16-bit model saved to {OUTPUT_DIR}/merged_16bit")

model.push_to_hub_merged(
    f"{HUB_MODEL_ID}-merged",
    tokenizer,
    save_method="merged_16bit",
    token=hf_token,
)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:27<01:27, 87.92s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:58<00:00, 59.25s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/quannguyen204/vimed-llama3.2-3b-sft-v1-merged`


# Inference Test

In [ ]:
# Inference Test
from unsloth import FastLanguageModel

# Enable inference mode (2x faster)
FastLanguageModel.for_inference(model)

# Test prompt
test_messages = [
    {
        "role": "system",
        "content": "Bạn là một trợ lý y tế ảo thông minh, với vai trò là một bác sĩ tư vấn trực tuyến chuyên nghiệp và tận tâm. Nhiệm vụ của bạn là giải đáp thắc mắc, câu hỏi về chủ đề y tế. Câu trả lời cần mang tính định hướng, giải thích nguyên nhân có thể, không được thay thế chẩn đoán của bệnh viện và phải luôn khuyên người dùng đến cơ sở y tế để có chẩn đoán chính xác."
    },
    {
        "role": "user",
        "content": "Những biểu hiện ngoài da nào có thể liên quan đến bệnh tiểu đường?"
    }
]

# Generate
inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=1500,
    use_cache = True, 
    temperature = 1.5, 
    min_p = 0.1
)

response = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("\n--- Model Response ---")
print(response)


--- Model Response ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Bạn là một trợ lý y tế ảo thông minh, với vai trò là một bác sĩ tư vấn trực tuyến chuyên nghiệp và tận tâm. Nhiệm vụ của bạn là giải đáp thắc mắc, câu hỏi về chủ đề y tế. Câu trả lời cần mang tính định hướng, giải thích nguyên nhân có thể, không được thay thế chẩn đoán của bệnh viện và phải luôn khuyên người dùng đến cơ sở y tế để có chẩn đoán chính xác.<|eot_id|><|start_header_id|>user<|end_header_id|>

Những biểu hiện ngoài da nào có thể liên quan đến bệnh tiểu đường?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Biểu hiện ngoài da thường gặp nhất ở bệnh nhân tiểu đường là vết mẩn đỏ ngứa có vảy ở da, đôi khi nổi nhiều ở chân, bàn chân. Nếu bị bệnh tiểu đường hơn 20 năm mà không được kiểm soát tốt thì xuất hiện sẹo do có bệnh viêm mạch máu nhỏ ở bên dưới. Bệnh nhân tiểu đường thường không bị bong tróc da sau khi rát do nhiệt. Đ

In [39]:
messages = [
    {
        "role": "system",
        "content": "Bạn là một trợ lý y tế ảo thông minh, với vai trò là một bác sĩ tư vấn trực tuyến chuyên nghiệp và tận tâm. Nhiệm vụ của bạn là giải đáp thắc mắc, câu hỏi về chủ đề y tế. Câu trả lời cần mang tính định hướng, giải thích nguyên nhân có thể, không được thay thế chẩn đoán của bệnh viện và phải luôn khuyên người dùng đến cơ sở y tế để có chẩn đoán chính xác."
    },
    {
        "role": "user",
        "content": "Những biểu hiện ngoài da nào có thể liên quan đến bệnh tiểu đường?"
    }
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1500, 
    temperature = 1.5, min_p = 0.1,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Cần lưu ý các triệu chứng ngoài da:
- Một vết hở da bị nhiễm trùng: đây cũng là biểu hiện của bệnh tiểu đường, bệnh nhân cần thăm khám, làm xét nghiệm lại để chẩn đoán nguyên nhân.
- Nang phì đại bàng quang: người bệnh tiểu đường nên đi kiểm tra và kiểm soát các yếu tố gây ra bệnh tiểu đường như hút thuốc lá.
- Viêm nhiễm da thường xuyên: nhiễm trùng hở vết loét kéo dài, thường liên quan đến bệnh tiểu đường, do người bệnh dễ mắc phải các bệnh lý do nhiễm trùng ở khớp, cổ tay hoặc mưng mủ từ vết thương do tiểu đường, viêm hạch.
- Có sưng, chảy mủ tại chân, tay hoặc ngón chân.
- Kém tích lũy các protein, tế bào bạch cầu ở nang sợi.
- Viêm tĩnh mạch thường là một triệu chứng phổ biến nhất ở bệnh nhân bị tiểu đường.
- Bệnh nang sợi ở chân có nhiều trường hợp xuất hiện kèm theo bệnh tiểu đường, đây là biểu hiện của bệnh lý tuần hoàn mạn tính, viêm và các bệnh tiểu đường.<|eot_id|>


In [ ]:
wandb.finish()

print("\n" + "="*60)
print("SFT Training Complete!")
print(f"Model: {HUB_MODEL_ID}")
print("="*60)


SFT Training Complete!
Model: quannguyen204/vimed-llama3.2-3b-sft-v1
